# EX5 Bending Forward Reconstruction with LSTM-based RT-RPINN

- This notebook trains the forward LSTM-based RT-RPINN field model for the EX5 bending shape-memory cycle.
- The recurrent backbone reconstructs the full displacement sequence from spatial coordinates, time, and temperature.

Run the cells in order. The paths are kept consistent with the original EX5 Python scripts.


## Script Notes

Example 5 (FORWARD, LSTM) — full-field reconstruction of the bending shape-memory
cycle with an LSTM-PINN (sequence-aware temporal backbone).

LSTM counterpart of forward_field_pinn_ex5.py. The displacement field is fit
DIRECTLY to the FE data (a pure regression): for each spatial point the LSTM
processes the temporal sequence of inputs (spatial features + time + temperature)
and outputs u(t) over the whole 4-step cycle, the recurrent state carrying the
loading/cooling/recovery history.

Same large-deformation lessons as the MLP forward:
  - SPATIAL Fourier features (cure spectral bias for the ~270° curl),
  - OUTPUT SCALING (predict O(1) × max|u| ≈ 119 mm),
  - pure displacement fit (no small-strain strain/stress, invalid under rotation).

Observables: no RF here; RM/UR are validation-only and NOT used.

Outputs (EX5/):
  ex5_lstm_forward_loss_history.csv
  ex5_lstm_forward_training_history.png
  ex5_lstm_forward_model.pth
  ex5_lstm_forward_training_log.txt


In [ ]:
# Notebook path setup
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'EX-5-RESULTS').exists() and (NOTEBOOK_DIR / 'EX5' / 'EX-5-RESULTS').exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / 'EX5'
NOTEBOOK_DIR = NOTEBOOK_DIR.resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
print(f'EX5 notebook directory: {NOTEBOOK_DIR}')


## Imports and definitions


In [ ]:
import sys
import math
import argparse
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from inverse_spatiotemporal_pinn_ex5 import (
    _Tee, device, FEDataLoader, set_global_seed,
)


## LSTM forward model


In [ ]:
# ---------------------------------------------------------------------------
class LSTMForwardEX5(nn.Module):
    """
    Per spatial point: encode (x,y,z, 45° fiber-local, spatial Fourier) → features,
    broadcast over time, append time-Fourier + temperature, run an LSTM over the
    temporal sequence, decode to u(t). Output is scaled by `output_scale` (≈ max|u|).
    """

    def __init__(self, spatial_hidden=(128, 128), lstm_hidden=128, lstm_layers=2,
                 n_fourier_space=5, n_fourier_time=5, output_scale=1.0, dropout=0.0):
        super().__init__()
        c = math.cos(math.radians(45.0)); s = math.sin(math.radians(45.0))
        self.register_buffer("cos_theta", torch.tensor(c, dtype=torch.float32))
        self.register_buffer("sin_theta", torch.tensor(s, dtype=torch.float32))
        self.register_buffer("sfreqs",
            2.0 * math.pi * torch.arange(1, n_fourier_space + 1, dtype=torch.float32))
        self.register_buffer("tfreqs",
            2.0 * math.pi * torch.arange(1, n_fourier_time + 1, dtype=torch.float32))
        self.register_buffer("output_scale",
            torch.tensor(float(output_scale), dtype=torch.float32))


## Spatial encoder: 5 raw (x,y,z,x_local,y_local) + 3*2*n_fs Fourier → features


In [ ]:
spatial_in = 5 + 3 * 2 * n_fourier_space
        dims = [spatial_in, *spatial_hidden]
        enc = []
        for i in range(len(dims) - 1):
            enc.append(nn.Linear(dims[i], dims[i + 1]))
            enc.append(nn.Tanh())
        self.spatial_encoder = nn.Sequential(*enc)


## LSTM input: spatial features + time-Fourier (2*n_ft+1) + temperature (1)


In [ ]:
lstm_in = dims[-1] + (2 * n_fourier_time + 1) + 1
        self.lstm = nn.LSTM(input_size=lstm_in, hidden_size=lstm_hidden,
                            num_layers=lstm_layers, batch_first=True,
                            dropout=dropout if lstm_layers > 1 else 0.0)
        self.decoder = nn.Sequential(
            nn.Linear(lstm_hidden, lstm_hidden), nn.Tanh(),
            nn.Linear(lstm_hidden, 3),
        )

    def _spatial_feats(self, x, y, z):
        """x,y,z: (B,1) normalised coords → (B, F) spatial features."""
        x_local = x * self.cos_theta + y * self.sin_theta
        y_local = -x * self.sin_theta + y * self.cos_theta
        feats = [x, y, z, x_local, y_local]
        for v in (x, y, z):
            for f in self.sfreqs:
                feats.append(torch.sin(v * f)); feats.append(torch.cos(v * f))
        return self.spatial_encoder(torch.cat(feats, dim=1))

    def forward(self, x, y, z, t_seq, T_seq):
        """
        Args:
            x, y, z: (B,1) normalised node coords
            t_seq, T_seq: (S,) normalised time / temperature for the sequence
        Returns:
            u: (B, S, 3) displacement in mm
        """
        B = x.shape[0]; S = t_seq.shape[0]
        sf = self._spatial_feats(x, y, z).unsqueeze(1).expand(B, S, -1)  # (B,S,F)
        t = t_seq.view(1, S, 1).expand(B, S, 1)
        t_enc = torch.cat(
            [torch.sin(t * f) for f in self.tfreqs] +
            [torch.cos(t * f) for f in self.tfreqs] + [t], dim=-1)
        T = T_seq.view(1, S, 1).expand(B, S, 1)
        inp = torch.cat([sf, t_enc, T], dim=-1)
        out, _ = self.lstm(inp)
        return self.decoder(out) * self.output_scale


## Build LSTM training tensors [N_nodes, N_times, ...] from the FE loader


In [ ]:
# ---------------------------------------------------------------------------
def build_sequence_arrays(fe_loader):
    fd = fe_loader.full_data
    node_col = None
    for c in ['NodeLabel', 'Node', 'NID']:
        if c in fd.columns:
            node_col = c; break
    times = np.sort(fd['Time'].unique())
    nodes = np.sort(fd[node_col].unique())
    n_nodes, n_times = len(nodes), len(times)
    sort_cols = [c for c in ['Step', 'Frame'] if c in fd.columns]

    def slice_dedup(t):


## (Step, Frame) state for each node to get one stable snapshot per time.


In [ ]:
df = fd[np.isclose(fd['Time'].values, t)]
        if sort_cols:
            df = df.sort_values(sort_cols)
        return df.drop_duplicates(subset=node_col, keep='last')


## Reference positions (earliest frame) and per-time temperature.


In [ ]:
first = slice_dedup(times[0]).set_index(node_col).reindex(nodes)
    ref = first[['X', 'Y', 'Z']].fillna(0.0).values.astype(np.float32)


## Displacement targets U[node, time, 3] + per-time temperature.


In [ ]:
U = np.zeros((n_nodes, n_times, 3), dtype=np.float32)
    temps = np.zeros(n_times, dtype=np.float32)
    for ti, t in enumerate(times):
        d = slice_dedup(t)
        temps[ti] = float(d['Temperature'].iloc[0])
        sl = d.set_index(node_col).reindex(nodes)
        U[:, ti, :] = sl[['U1', 'U2', 'U3']].fillna(0.0).values
    return ref, times.astype(np.float32), temps, U


def main():
    parser = argparse.ArgumentParser(
        description="EX5 bending FORWARD field LSTM-PINN with fixed material (pure data fit).")
    parser.add_argument('--epochs', type=int, default=10000)
    parser.add_argument('--batch-size', type=int, default=256, help='Spatial nodes per step.')
    parser.add_argument('--lr', type=float, default=2e-3)
    parser.add_argument('--lstm-hidden', type=int, default=128)
    parser.add_argument('--lstm-layers', type=int, default=2)
    parser.add_argument('--stride', type=int, default=5)
    parser.add_argument('--n-temporal', type=int, default=150)
    parser.add_argument('--n-spatial', type=int, default=3000)
    parser.add_argument('--seed', type=int, default=42)
    args = parser.parse_args(args=[])
    set_global_seed(args.seed)
    n_spatial = args.n_spatial if args.n_spatial and args.n_spatial > 0 else None
    n_temporal = args.n_temporal if args.n_temporal and args.n_temporal > 0 else None

    print("=" * 70)
    print("EX5 FORWARD FIELD RECONSTRUCTION — LSTM-PINN")
    print("Bending shape-memory cycle (95x13x2 beam, end rotation UR=4.71 rad)")
    print("Pure displacement fit (no small-strain strain/stress; material fixed)")
    print("=" * 70)
    print(f"Seed: {args.seed}")

    script_dir = NOTEBOOK_DIR
    print("\nLoading FE data...")
    fe_loader = FEDataLoader(
        script_dir / 'EX-5-RESULTS',
        script_dir / 'ex-5-Bending-RM.csv',
        script_dir / 'ex-5-Bending-UR.csv',
        script_dir / 'frame-time.csv',
        stride=args.stride, n_spatial=n_spatial, spatial_seed=args.seed,
        n_temporal=n_temporal,
    )
    fe_loader.load_all_data()
    bounds = fe_loader.get_domain_bounds()

    ref, times, temps, U = build_sequence_arrays(fe_loader)
    print(f"\nSequence arrays: nodes={ref.shape[0]}  times={times.shape[0]}")


## Normalisation (matches the MLP solver's norm_coords)


In [ ]:
x_min, y_min, z_min = bounds['x_min'], bounds['y_min'], bounds['z_min']
    Lx = bounds['x_max'] - x_min; Ly = bounds['y_max'] - y_min; Lz = bounds['z_max'] - z_min
    t_min = bounds['t_min']; Lt = max(bounds['t_max'] - t_min, 1e-9)
    T_min = bounds['T_min']; LT = max(bounds['T_max'] - T_min, 1.0)

    xn = ((ref[:, 0] - x_min) / Lx).reshape(-1, 1)
    yn = ((ref[:, 1] - y_min) / Ly).reshape(-1, 1)
    zn = ((ref[:, 2] - z_min) / Lz).reshape(-1, 1)
    t_hat = ((times - t_min) / Lt)
    T_hat = ((temps - T_min) / LT)

    ft = torch.float32
    xt = torch.tensor(xn, dtype=ft, device=device)
    yt = torch.tensor(yn, dtype=ft, device=device)
    zt = torch.tensor(zn, dtype=ft, device=device)
    t_seq = torch.tensor(t_hat, dtype=ft, device=device)
    T_seq = torch.tensor(T_hat, dtype=ft, device=device)
    U_t = torch.tensor(U, dtype=ft, device=device)     # (N, S, 3) targets in mm

    u_mag = np.linalg.norm(U.reshape(-1, 3), axis=1)
    u_ref = float(u_mag.max())
    u_ref_sq = (u_ref + 1e-10) ** 2
    output_scale = 1.1 * u_ref
    print(f"Output displacement scale: {output_scale:.2f} mm (1.1 x max|u|)")

    model = LSTMForwardEX5(
        lstm_hidden=args.lstm_hidden, lstm_layers=args.lstm_layers,
        output_scale=output_scale,
    ).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"LSTM-PINN parameters: {n_params:,}")

    optimizer = Adam(model.parameters(), lr=args.lr)
    scheduler = CosineAnnealingLR(optimizer, T_max=args.epochs, eta_min=1e-5)

    n_nodes = xt.shape[0]
    loss_history = []
    print(f"\nStarting LSTM forward training ({args.epochs} epochs)...")
    print("-" * 70)
    for epoch in range(args.epochs):
        model.train()
        idx = torch.randint(0, n_nodes, (min(args.batch_size, n_nodes),), device=device)
        u_pred = model(xt[idx], yt[idx], zt[idx], t_seq, T_seq)   # (B,S,3)
        loss = torch.mean((u_pred - U_t[idx]) ** 2) / u_ref_sq

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()
        scheduler.step()
        loss_history.append(loss.item())

        if epoch % 100 == 0 or epoch == args.epochs - 1:
            print(f"[Epoch {epoch:5d}] data={loss.item():.4e}  "
                  f"lr={optimizer.param_groups[0]['lr']:.2e}")


## Save


In [ ]:
print("\nSaving results...")
    import csv
    with open(script_dir / 'ex5_lstm_forward_loss_history.csv', 'w', newline='') as f:
        w = csv.writer(f); w.writerow(['epoch', 'data'])
        for i, v in enumerate(loss_history):
            w.writerow([i, v])

    plt.figure(figsize=(7, 5))
    plt.semilogy(loss_history, 'b-', lw=1.5)
    plt.xlabel('Epoch'); plt.ylabel('Data loss (normalised)')
    plt.title('EX5 LSTM forward — training loss')
    plt.grid(True, alpha=0.3); plt.tight_layout()
    plt.savefig(script_dir / 'ex5_lstm_forward_training_history.png', dpi=200)
    plt.close()

    torch.save({
        'model_state_dict': model.state_dict(),
        'output_scale': output_scale,
        'bounds': bounds,
        'arch': {'lstm_hidden': args.lstm_hidden, 'lstm_layers': args.lstm_layers},
    }, script_dir / 'ex5_lstm_forward_model.pth')
    print(f"Model saved. Final data loss = {loss_history[-1]:.4e}")
    print("\n" + "=" * 70)
    print("EX5 LSTM FORWARD RECONSTRUCTION COMPLETE")
    print("=" * 70)


if __name__ == "__main__":
    _script_dir = NOTEBOOK_DIR
    _log_path = _script_dir / 'ex5_lstm_forward_training_log.txt'
    _tee = _Tee(sys.stdout, _log_path)
    sys.stdout = _tee
    try:
        main()
    finally:
        sys.stdout = _tee._orig
        _tee.close()
        _tee._orig.write(f"\nLog saved to: {_log_path}\n")
